# 1000-hand simulation: hero follows the action advisor

- **Hero** only acts according to `action_advisor.predict_decision_advisor` (fold / call / raise).
- **Raises:** when the advisor says **raise**, the simulation picks a **raise-to** total `min(min_raise + extra, max_raise)` where `extra` is **5** on the hero’s first raise of the hand, **10** on the second, **20** on the third, … (`5 × 2^k`). `min_raise` / `max_raise` come from the live engine (`build_view`), so sizing stays legal.
- Bots use the same `poker_core.run_one_bot_turn` stack as the live app.

Run the code cell from the repo root **or** from `notebooks/` (path detection handles both).

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd


def _poker_page_dir() -> Path:
    c = Path.cwd().resolve()
    for p in (c / "poker_page", c.parent / "poker_page"):
        if p.is_dir():
            return p.resolve()
    raise FileNotFoundError("Could not find poker_page/ (cwd should be repo root or notebooks/).")


PP = _poker_page_dir()
if str(PP) not in sys.path:
    sys.path.insert(0, str(PP))

import poker_core  # noqa: E402
from game_state import GameState  # noqa: E402

N_HANDS = 1000
MAX_STEPS_PER_HAND = 8000


def choose_raise_to_amount(gs, view: dict, hero_raise_index: int) -> int | None:
    """Return legal raise-to total, or None if no raise is legal."""
    state = gs.poker_state
    if state is None:
        return None
    min_r = int(view["min_raise"])
    max_r = int(view["max_raise"])
    extra = 5 * (2**hero_raise_index)
    want = min(max_r, max(min_r, min_r + extra))
    for cand in (want, max_r, min_r):
        if state.can_complete_bet_or_raise_to(int(cand)):
            return int(cand)
    return None


def play_one_hand(gs: GameState) -> dict:
    """Play until terminal; return row dict of outcomes."""
    hero = int(gs.hero)
    start_stack = int((gs.master_bankrolls or [0])[hero])
    hero_raise_idx = 0
    steps = 0
    folds = 0
    calls = 0
    raises = 0
    err_raise_fallback = 0

    while steps < MAX_STEPS_PER_HAND:
        steps += 1
        poker_core.run_dealer(gs)
        ps = gs.poker_state
        if ps is None or not bool(getattr(ps, "status", True)):
            poker_core.capture_terminal_if_needed(gs)
            break
        ti = poker_core.live_turn_seat(gs)
        if ti is None:
            poker_core.capture_terminal_if_needed(gs)
            break
        if ti == hero:
            view = poker_core.build_view(gs)
            rec = (view.get("action_advisor_action") or "call").lower()
            if rec == "fold":
                poker_core.apply_hero_action(gs, {"type": "fold"})
                folds += 1
            elif rec == "call":
                poker_core.apply_hero_action(gs, {"type": "call"})
                calls += 1
            elif rec == "raise":
                amt = choose_raise_to_amount(gs, view, hero_raise_idx)
                if amt is not None:
                    poker_core.apply_hero_action(gs, {"type": "raise", "amount": amt})
                    if gs.action_error:
                        gs.action_error = ""
                        poker_core.apply_hero_action(gs, {"type": "call"})
                        err_raise_fallback += 1
                    else:
                        raises += 1
                        hero_raise_idx += 1
                else:
                    poker_core.apply_hero_action(gs, {"type": "call"})
                    err_raise_fallback += 1
            else:
                poker_core.apply_hero_action(gs, {"type": "call"})
                calls += 1
            poker_core.run_dealer(gs)
        else:
            poker_core.run_one_bot_turn(gs)

    end_stack = int((gs.master_bankrolls or [0])[hero])
    pay = poker_core.build_view(gs).get("payoffs") or []
    hero_pay = int(pay[hero]) if hero < len(pay) else 0
    return {
        "hand_steps": steps,
        "hero_start_stack": start_stack,
        "hero_end_stack": end_stack,
        "hero_net_chips": end_stack - start_stack,
        "hero_payoff_last_view": hero_pay,
        "hero_folds": folds,
        "hero_calls": calls,
        "hero_raises": raises,
        "illegal_raise_fallbacks": err_raise_fallback,
    }


gs = GameState()
gs.hero = 0
poker_core.ensure_initialized(gs)

rows: list[dict] = []
for h in range(N_HANDS):
    if h > 0:
        poker_core.new_hand(gs, advance_button=True)
    rows.append(play_one_hand(gs))

df = pd.DataFrame(rows)
print(df.describe().T)
print("\nCumulative hero net chips:", int(df["hero_net_chips"].sum()))
print("Mean net per hand:", float(df["hero_net_chips"].mean()))
print(
    "Advisor mix (approx):",
    {
        "folds": int(df["hero_folds"].sum()),
        "calls": int(df["hero_calls"].sum()),
        "raises": int(df["hero_raises"].sum()),
        "raise→call_fallbacks": int(df["illegal_raise_fallbacks"].sum()),
    },
)
df.head(10)

                          count      mean          std    min      25%  \
hand_steps               1000.0    25.670     4.665712   13.0    22.00   
hero_start_stack         1000.0  2792.756  1927.057087   38.0  1053.50   
hero_end_stack           1000.0  2794.240  1925.629318   38.0  1059.75   
hero_net_chips           1000.0     1.484   153.803053 -784.0   -24.00   
hero_payoff_last_view    1000.0     1.484   153.803053 -784.0   -24.00   
hero_folds               1000.0     0.214     0.410332    0.0     0.00   
hero_calls               1000.0     4.901     1.777177    0.0     4.00   
hero_raises              1000.0     0.065     0.359032    0.0     0.00   
illegal_raise_fallbacks  1000.0     0.000     0.000000    0.0     0.00   

                            50%      75%     max  
hand_steps                 25.0    29.00    42.0  
hero_start_stack         2103.5  4521.75  6238.0  
hero_end_stack           2103.5  4521.75  6238.0  
hero_net_chips             -4.0    -2.00  2027.0  
hero

,hand_steps,hero_start_stack,hero_end_stack,hero_net_chips,hero_payoff_last_view,hero_folds,hero_calls,hero_raises,illegal_raise_fallbacks
0,28,200,536,336,336,0,7,0,0
1,22,536,544,8,8,0,4,0,0
2,26,544,542,-2,-2,1,2,0,0
3,23,542,434,-108,-108,0,8,0,0
4,25,434,350,-84,-84,0,7,0,0
5,31,350,348,-2,-2,1,2,0,0
6,28,348,281,-67,-67,0,7,0,0
7,23,281,390,109,109,0,6,0,0
8,33,390,366,-24,-24,1,5,0,0
9,32,366,302,-64,-64,1,7,0,0
